# NLP Lab Task 5 — N-gram Language Models
# Name : Shehzad Aman
# Instructor : Sir Zaeem Akbar
Build unigram, bigram and trigram language models from scratch: tokenization, counting, MLE probabilities, smoothing, and perplexity. Standard library only.

In [1]:
import math
import itertools
from collections import Counter

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test = ["the cat sat on the rug", "the fish sat on the mat"]

# V = all training words + </s>  (<s> is a context marker only, never a vocabulary word)

## Task 1: Tokenization

Lowercase, split on spaces, wrap with `<s>` / `</s>`.

In [2]:
# 1. tokenize a sentence
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]

# 2. apply to every sentence
train_tok = [tokenize(s) for s in train]
test_tok = [tokenize(s) for s in test]
print("Train tokens:")
for t in train_tok:
    print(" ", t)
print("Test tokens:")
for t in test_tok:
    print(" ", t)

# 3. token counts with / without markers
with_markers = sum(len(t) for t in train_tok)
without_markers = sum(len(t) - 2 for t in train_tok)
print("\nTotal training tokens (with markers):", with_markers)
print("Total training tokens (without markers):", without_markers)

# 4. vocabulary
def build_vocab(sentences):
    vocab = set()
    for s in sentences:
        vocab.update(tokenize(s)[1:])   # drop leading <s>, keep </s>
    return vocab

vocab = build_vocab(train)
V = len(vocab)
print("\nV =", V)
print("Vocabulary:", sorted(vocab))

Train tokens:
  ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
  ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
  ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
  ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
Test tokens:
  ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
  ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Total training tokens (with markers): 30
Total training tokens (without markers): 22

V = 11
Vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2: Generating N-grams

Sliding window over a token list. Returns `[]` instead of crashing when `n` is larger than the sentence.

In [3]:
# 1. generic n-gram generator
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

# 2. uni/bi/trigrams of the first training sentence
first = train_tok[0]
print("Unigrams:", get_ngrams(first, 1))
print("Bigrams :", get_ngrams(first, 2))
print("Trigrams:", get_ngrams(first, 3))

# 3. m tokens -> m - n + 1 ngrams, for every sentence and n = 1, 2, 3
for t in train_tok + test_tok:
    for n in (1, 2, 3):
        assert len(get_ngrams(t, n)) == len(t) - n + 1
print("\nm - n + 1 check: OK for every sentence, n = 1, 2, 3")

# 4. n larger than the sentence -> empty list, no crash
print("get_ngrams(first, 50) ->", get_ngrams(first, 50))

Unigrams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
Bigrams : [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
Trigrams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]

m - n + 1 check: OK for every sentence, n = 1, 2, 3
get_ngrams(first, 50) -> []


## Task 3: Counting N-grams

`Counter` over every sentence's n-grams. Context counts `C(v)` and `C(w1,w2)` come straight from the bigram/trigram tables.

In [4]:
# 1. unigram / bigram / trigram counts (unigram counts exclude <s>)
unigram_counts = Counter(t[0] for s in train for t in get_ngrams(tokenize(s)[1:], 1))
bigram_counts = Counter(t for s in train for t in get_ngrams(tokenize(s), 2))
trigram_counts = Counter(t for s in train for t in get_ngrams(tokenize(s), 3))

# 2. tables, highest count first
print("Unigrams:", unigram_counts.most_common())
print("\nBigrams:", bigram_counts.most_common())
print("\nTrigrams:", trigram_counts.most_common())

# 3. top 5 bigrams + distinct counts
print("\nTop 5 bigrams:", bigram_counts.most_common(5))
print("Distinct unigrams/bigrams/trigrams:", len(unigram_counts), len(bigram_counts), len(trigram_counts))

# 4. safe lookup, 0 for unseen
def count_of(ngram):
    table = {1: unigram_counts, 2: bigram_counts, 3: trigram_counts}[len(ngram)]
    key = ngram[0] if len(ngram) == 1 else ngram
    return table.get(key, 0)

print("\ncount_of(('the','cat')) =", count_of(("the", "cat")), "(seen)")
print("count_of(('cat','dog'))  =", count_of(("cat", "dog")), "(unseen)")

# context counts, derived from the tables above (used by every model below)
prev_counts = Counter()
for (a, b), c in bigram_counts.items():
    prev_counts[a] += c

pair_counts = Counter()
for (a, b, c3), c in trigram_counts.items():
    pair_counts[(a, b)] += c

Unigrams: [('the', 8), ('</s>', 4), ('cat', 2), ('sat', 2), ('on', 2), ('ate', 2), ('dog', 2), ('mat', 1), ('fish', 1), ('rug', 1), ('bone', 1)]

Bigrams: [(('<s>', 'the'), 4), (('the', 'cat'), 2), (('sat', 'on'), 2), (('on', 'the'), 2), (('ate', 'the'), 2), (('the', 'dog'), 2), (('cat', 'sat'), 1), (('the', 'mat'), 1), (('mat', '</s>'), 1), (('cat', 'ate'), 1), (('the', 'fish'), 1), (('fish', '</s>'), 1), (('dog', 'sat'), 1), (('the', 'rug'), 1), (('rug', '</s>'), 1), (('dog', 'ate'), 1), (('the', 'bone'), 1), (('bone', '</s>'), 1)]

Trigrams: [(('<s>', 'the', 'cat'), 2), (('sat', 'on', 'the'), 2), (('<s>', 'the', 'dog'), 2), (('the', 'cat', 'sat'), 1), (('cat', 'sat', 'on'), 1), (('on', 'the', 'mat'), 1), (('the', 'mat', '</s>'), 1), (('the', 'cat', 'ate'), 1), (('cat', 'ate', 'the'), 1), (('ate', 'the', 'fish'), 1), (('the', 'fish', '</s>'), 1), (('the', 'dog', 'sat'), 1), (('dog', 'sat', 'on'), 1), (('on', 'the', 'rug'), 1), (('the', 'rug', '</s>'), 1), (('the', 'dog', 'ate'), 1), 

## Task 4: Unigram Model

MLE: `P(w) = C(w) / N`. A full-sentence product of several small probabilities can round to `0.0000` at 4 decimals — that's normal underflow, not the zero-probability problem (Task 7 is about a *true* zero from an unseen n-gram).

In [5]:
N_uni = sum(unigram_counts.values())

# 1. MLE unigram probability
def unigram_prob(w):
    return unigram_counts.get(w, 0) / N_uni

# 2. probability table, must sum to 1
probs = {w: round(unigram_prob(w), 4) for w in sorted(vocab)}
print("Unigram probabilities:", probs)
total = sum(unigram_prob(w) for w in vocab)
print("Sum of probabilities =", total)
assert abs(total - 1.0) < 1e-9

# 3. sentence probability = product over all words (order-independent)
def unigram_sentence_prob(sentence):
    p = 1.0
    for w in tokenize(sentence)[1:]:
        p *= unigram_prob(w)
    return p

for s in test:
    p = unigram_sentence_prob(s)
    print(f"\nP('{s}') = {p:.4e}  (rounded: {round(p, 4)})")

# 4. word order does not matter for a unigram model
words0 = test[0].split()
distinct_probs = {round(unigram_sentence_prob(" ".join(perm)), 4) for perm in itertools.permutations(words0)}
print("\nDistinct probabilities across all word orders of test[0]:", distinct_probs)
assert len(distinct_probs) == 1

Unigram probabilities: {'</s>': 0.1538, 'ate': 0.0769, 'bone': 0.0385, 'cat': 0.0769, 'dog': 0.0769, 'fish': 0.0385, 'mat': 0.0385, 'on': 0.0769, 'rug': 0.0385, 'sat': 0.0769, 'the': 0.3077}
Sum of probabilities = 1.0

P('the cat sat on the rug') = 2.5499e-07  (rounded: 0.0)

P('the fish sat on the mat') = 1.2749e-07  (rounded: 0.0)

Distinct probabilities across all word orders of test[0]: {0.0}


## Task 5: Bigram Model (MLE)

`P(w | v) = C(v, w) / C(v)`. Sentence probability uses the chain rule starting at `<s>`.

In [6]:
# 1. MLE bigram probability
def bigram_prob(prev, w):
    c = prev_counts.get(prev, 0)
    return bigram_counts.get((prev, w), 0) / c if c else 0.0

# 2. next-word distribution, must sum to 1
def next_word_distribution(prev):
    return {w: bigram_prob(prev, w) for w in sorted(vocab)}

dist_the = next_word_distribution("the")
print("Next word after 'the':", {w: round(p, 4) for w, p in dist_the.items()})
print("Sum =", sum(dist_the.values()))
assert abs(sum(dist_the.values()) - 1.0) < 1e-9

# 3. sentence probability via the chain rule
def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    p = 1.0
    for i in range(1, len(tokens)):
        p *= bigram_prob(tokens[i - 1], tokens[i])
    return p

for s in test:
    print(f"P('{s}') = {round(bigram_sentence_prob(s), 4)}")

# 4. most likely next word, ties broken alphabetically
def predict_next(prev):
    dist = next_word_distribution(prev)
    return min(dist, key=lambda w: (-dist[w], w))

print()
for prev in ["the", "cat", "dog", "sat", "ate"]:
    print(f"predict_next('{prev}') -> {predict_next(prev)}")

Next word after 'the': {'</s>': 0.0, 'ate': 0.0, 'bone': 0.125, 'cat': 0.25, 'dog': 0.25, 'fish': 0.125, 'mat': 0.125, 'on': 0.0, 'rug': 0.125, 'sat': 0.0, 'the': 0.0}
Sum = 1.0
P('the cat sat on the rug') = 0.0156
P('the fish sat on the mat') = 0.0

predict_next('the') -> cat
predict_next('cat') -> ate
predict_next('dog') -> ate
predict_next('sat') -> on
predict_next('ate') -> the


## Task 6: Trigram Model

`P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2)`. The first word (right after `<s>`) uses the bigram model; every later word uses the trigram model.

In [7]:
# 1. MLE trigram probability
def trigram_prob(w1, w2, w3):
    c = pair_counts.get((w1, w2), 0)
    return trigram_counts.get((w1, w2, w3), 0) / c if c else 0.0

# 2. sentence probability: bigram for the first word, trigram after that
def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    p = bigram_prob(tokens[0], tokens[1])
    for i in range(2, len(tokens)):
        p *= trigram_prob(tokens[i - 2], tokens[i - 1], tokens[i])
    return p

for s in test:
    print(f"P('{s}') = {round(trigram_sentence_prob(s), 4)}")

# 3. distinct n-gram counts
print("\n{:<10}{:>6}".format("model", "count"))
for name, table in [("unigram", unigram_counts), ("bigram", bigram_counts), ("trigram", trigram_counts)]:
    print(f"{name:<10}{len(table):>6}")

# 4. unseen bigrams vs unseen trigrams, per test sentence
def unseen_ngrams(sentence, n, table):
    return [g for g in get_ngrams(tokenize(sentence), n) if table.get(g, 0) == 0]

print()
for s in test:
    ub = unseen_ngrams(s, 2, bigram_counts)
    ut = unseen_ngrams(s, 3, trigram_counts)
    more = "trigram" if len(ut) > len(ub) else "bigram" if len(ub) > len(ut) else "tie"
    print(f"'{s}'\n  unseen bigrams  ({len(ub)}): {ub}\n  unseen trigrams ({len(ut)}): {ut}\n  more unseen: {more}")

P('the cat sat on the rug') = 0.125
P('the fish sat on the mat') = 0.0

model      count
unigram       11
bigram        18
trigram       19

'the cat sat on the rug'
  unseen bigrams  (0): []
  unseen trigrams (0): []
  more unseen: tie
'the fish sat on the mat'
  unseen bigrams  (1): [('fish', 'sat')]
  unseen trigrams (3): [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
  more unseen: trigram


## Task 7: Zero-Probability Problem

One unseen bigram makes the whole MLE sentence probability exactly `0`. `safe_log` turns that into `-inf` instead of a crash.

In [8]:
# 1. bigrams of a sentence that never occur in training
def unseen_bigrams(sentence):
    return unseen_ngrams(sentence, 2, bigram_counts)

# 2. MLE probability of each, and the full sentence probability
for s in test:
    ub = unseen_bigrams(s)
    sp = bigram_sentence_prob(s)
    print(f"'{s}'  unseen bigrams: {ub}")
    for bg in ub:
        print(f"   MLE P{bg} = {bigram_prob(bg[0], bg[1])}")
    print(f"   sentence probability = {round(sp, 4)}" + ("  <- exactly 0" if sp == 0 else ""))

# 3. how many possible word pairs never occur in training (markers excluded)
words_only = sorted(vocab - {"</s>"})
possible_pairs = [(a, b) for a in words_only for b in words_only]
never_seen = [p for p in possible_pairs if bigram_counts.get(p, 0) == 0]
pct = len(never_seen) / len(possible_pairs) * 100
print(f"\n{len(never_seen)} / {len(possible_pairs)} word pairs never seen = {pct:.2f}%")

# 4. safe_log avoids math.log(0) crashing
def safe_log(p):
    return float("-inf") if p == 0 else math.log(p)

p2 = bigram_sentence_prob(test[1])
print(f"\nlog P('{test[1]}') = {safe_log(p2)}")

'the cat sat on the rug'  unseen bigrams: []
   sentence probability = 0.0156
'the fish sat on the mat'  unseen bigrams: [('fish', 'sat')]
   MLE P('fish', 'sat') = 0.0
   sentence probability = 0.0  <- exactly 0

87 / 100 word pairs never seen = 87.00%

log P('the fish sat on the mat') = -inf


## Task 8: Smoothing

Add-one and add-k smoothing move a little probability mass from seen bigrams to unseen ones, so no bigram is ever exactly 0.

In [9]:
# 1. add-one (Laplace) smoothing
def bigram_prob_addone(prev, w):
    return (bigram_counts.get((prev, w), 0) + 1) / (prev_counts.get(prev, 0) + V)

# 2. MLE vs add-one, side by side
for prev, w in [("fish", "sat"), ("the", "cat")]:
    mle = round(bigram_prob(prev, w), 4)
    addone = round(bigram_prob_addone(prev, w), 4)
    print(f"P({w}|{prev})  MLE = {mle}   add-one = {addone}")

# 3. the smoothed distribution still sums to 1, for every possible previous word
for prev in sorted(vocab):
    total = sum(bigram_prob_addone(prev, w) for w in sorted(vocab))
    assert abs(total - 1.0) < 1e-9
print("\nadd-one next-word distribution sums to 1 for every prev word: OK")

# 4. add-k smoothing, k = 1, 0.5, 0.1, 0.01
def bigram_prob_addk(prev, w, k):
    return (bigram_counts.get((prev, w), 0) + k) / (prev_counts.get(prev, 0) + k * V)

print("\n{:<6}{:>12}".format("k", "P(sat|fish)"))
for k in [1, 0.5, 0.1, 0.01]:
    print(f"{k:<6}{round(bigram_prob_addk('fish', 'sat', k), 4):>12}")

P(sat|fish)  MLE = 0.0   add-one = 0.0833
P(cat|the)  MLE = 0.25   add-one = 0.1579

add-one next-word distribution sums to 1 for every prev word: OK

k      P(sat|fish)
1           0.0833
0.5         0.0769
0.1         0.0476
0.01         0.009


## Task 9: Perplexity and Model Comparison

Perplexity scores how well a model predicts unseen text — lower is better, and a single zero probability makes it infinite.

In [10]:
# 1. perplexity over a list of sentences
def perplexity(sentences, prob_fn):
    log_sum = 0.0
    n_tokens = 0
    for s in sentences:
        p = prob_fn(s)
        if p == 0:
            return float("inf")
        log_sum += math.log(p)
        n_tokens += len(tokenize(s)) - 1   # every token except the leading <s>
    return math.exp(-log_sum / n_tokens)

# helper sentence-probability functions for the models being compared
def bigram_sentence_prob_addone(sentence):
    tokens = tokenize(sentence)
    p = 1.0
    for i in range(1, len(tokens)):
        p *= bigram_prob_addone(tokens[i - 1], tokens[i])
    return p

def unigram_prob_addone(w):
    return (unigram_counts.get(w, 0) + 1) / (N_uni + V)

def unigram_sentence_prob_addone(sentence):
    p = 1.0
    for w in tokenize(sentence)[1:]:
        p *= unigram_prob_addone(w)
    return p

# 2. bigram MLE vs bigram add-one vs unigram add-one, on both test sentences together
print("{:<18}{:>12}".format("model", "perplexity"))
print("{:<18}{:>12}".format("bigram MLE", perplexity(test, bigram_sentence_prob)))
print("{:<18}{:>12}".format("bigram add-one", round(perplexity(test, bigram_sentence_prob_addone), 4)))
print("{:<18}{:>12}".format("unigram add-one", round(perplexity(test, unigram_sentence_prob_addone), 4)))

model               perplexity
bigram MLE                 inf
bigram add-one          5.7969
unigram add-one          9.137


In [11]:
# 3. add-k bigram perplexity, k = 1, 0.5, 0.1, 0.01
def bigram_sentence_prob_addk(sentence, k):
    tokens = tokenize(sentence)
    p = 1.0
    for i in range(1, len(tokens)):
        p *= bigram_prob_addk(tokens[i - 1], tokens[i], k)
    return p

ks = [1, 0.5, 0.1, 0.01]
pp_by_k = {k: perplexity(test, lambda s, k=k: bigram_sentence_prob_addk(s, k)) for k in ks}

print("{:<6}{:>12}".format("k", "perplexity"))
for k in ks:
    print(f"{k:<6}{round(pp_by_k[k], 4):>12}")
best_k = min(pp_by_k, key=pp_by_k.get)
print(f"\nbest k (tuned directly on the test set) = {best_k}, perplexity = {round(pp_by_k[best_k], 4)}")

# 4. proper validation: train on sentences 1-3, pick k on sentence 4, then score the real test set
train3 = train[:3]
val_sentence = train[3]

bigram_counts_3 = Counter(t for s in train3 for t in get_ngrams(tokenize(s), 2))
prev_counts_3 = Counter()
for (a, b), c in bigram_counts_3.items():
    prev_counts_3[a] += c

def bigram_prob_addk_3(prev, w, k):
    return (bigram_counts_3.get((prev, w), 0) + k) / (prev_counts_3.get(prev, 0) + k * V)

def bigram_sentence_prob_addk_3(sentence, k):
    tokens = tokenize(sentence)
    p = 1.0
    for i in range(1, len(tokens)):
        p *= bigram_prob_addk_3(tokens[i - 1], tokens[i], k)
    return p

val_pp = {k: perplexity([val_sentence], lambda s, k=k: bigram_sentence_prob_addk_3(s, k)) for k in ks}
print("\nvalidation perplexity (trained on sentences 1-3, scored on sentence 4):")
for k in ks:
    print(f"  k={k}: {round(val_pp[k], 4)}")

best_k_val = min(val_pp, key=val_pp.get)
test_pp = perplexity(test, lambda s: bigram_sentence_prob_addk_3(s, best_k_val))
print(f"\nbest k via validation = {best_k_val}")
print(f"test perplexity with that k (3-sentence training) = {round(test_pp, 4)}")
print(f"best k found directly on the test set (4-sentence training) = {best_k}, perplexity = {round(pp_by_k[best_k], 4)}")
print("\nNote: the two rows above don't use the same training size (3 vs 4 sentences),")
print("so this compares the *selection method*, not a perfectly controlled pair.")

k       perplexity
1           5.7969
0.5         4.5703
0.1         2.9559
0.01        2.6272

best k (tuned directly on the test set) = 0.01, perplexity = 2.6272

validation perplexity (trained on sentences 1-3, scored on sentence 4):
  k=1: 8.5857
  k=0.5: 8.0163
  k=0.1: 8.026
  k=0.01: 13.1304

best k via validation = 0.5
test perplexity with that k (3-sentence training) = 4.4539
best k found directly on the test set (4-sentence training) = 0.01, perplexity = 2.6272

Note: the two rows above don't use the same training size (3 vs 4 sentences),
so this compares the *selection method*, not a perfectly controlled pair.
